# RQ2 — Fed-CGM Sanity Check

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [10]:
import json

import numpy as np
import torch
from torch import nn
from src.train import set_seed

from src import dataloader as data
from src import federated
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.nn_baseline import GRUPredictor

In [8]:
# 1. Sanity-check configuration

SEED = 7
BATCH_SIZE = 256

LOCAL_EPOCHS = 1
MAX_ROUNDS = 5
LR = 1e-4
WEIGHT_DECAY = 1e-5
MAX_GRAD_NORM = 1.0

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

set_seed(SEED)

print(f"Device: {DEVICE}")
print(f"Seed: {SEED}")
print(f"Local epochs: {LOCAL_EPOCHS}")
print(f"Rounds: {MAX_ROUNDS}")


Device: cuda
Seed: 7
Local epochs: 1
Rounds: 5


## 2. Unit-check the FedAvg aggregation

In [11]:
# Locate the weighted-average helper used by federated.py.
agg_fn = getattr(federated, '_weighted_average_state_dicts', None)

if agg_fn is None:
    print("⚠ Could not access _weighted_average_state_dicts directly; skipping this private-function unit test.")
else:
    state_a = {'weight': torch.tensor([1.0])}
    state_b = {'weight': torch.tensor([3.0])}

    averaged = agg_fn([state_a, state_b], [1, 3])
    observed = float(averaged['weight'].item())
    expected = 2.5

    print("Expected:", expected)
    print("Observed:", observed)

    assert np.isclose(observed, expected), (
        f"FedAvg aggregation failed: expected {expected}, got {observed}"
    )
    print("✓ FedAvg weighted aggregation unit test passed.")


Expected: 2.5
Observed: 2.5
✓ FedAvg weighted aggregation unit test passed.


## 3. Build participant-level training clients


In [17]:
client_loaders = federated.load_federated_clients(
    split='train',
    encoder_name=None,
    batch_size=BATCH_SIZE,
)

n_clients = len(client_loaders)
client_sizes = {
    pid: len(loader.dataset)
    for pid, loader in client_loaders.items()
}

print("Number of training clients:", n_clients)
print("Total training samples:", sum(client_sizes.values()))
print("Smallest client:", min(client_sizes.values()))
print("Largest client:", max(client_sizes.values()))
print("Mean client size:", np.mean(list(client_sizes.values())))

assert n_clients == 30, f"Expected 30 training clients, got {n_clients}"
assert all(n > 0 for n in client_sizes.values()), "At least one client is empty."
assert len(set(client_loaders.keys())) == n_clients

print("30 non-empty participant-level clients constructed.")


Number of training clients: 30
Total training samples: 82681
Smallest client: 2552
Largest client: 2807
Mean client size: 2756.0333333333333
30 non-empty participant-level clients constructed.


## 4. Load validation data and train-only CGM normalization statistics

In [16]:
stats = data.load_cgm_stats()

val_loader = data.load_multimodal_tensors(
    'val',
    encoder_name=None,
    shuffle=False,
)

cgm_mean = stats['cgm_mean']
cgm_std = stats['cgm_std']

assert np.isfinite(cgm_mean)
assert np.isfinite(cgm_std)
assert cgm_std > 0

print("n_val_batches =", len(val_loader))
print("CGM mean:", cgm_mean)
print("CGM std:", cgm_std)
print("Validation loader and CGM normalization statistics loaded.")


n_val_batches = 64
CGM mean: 140.10437936402963
CGM std: 42.125711031429994
Validation loader and CGM normalization statistics loaded.


## 5. Instantiate the locked RQ1 GRU


In [18]:
def model_fn():
    return GRUPredictor(hidden_size=256, num_layers=3, dropout=0.2)

probe_model = model_fn()

n_params = sum(p.numel() for p in probe_model.parameters())
n_trainable = sum(p.numel() for p in probe_model.parameters() if p.requires_grad)

print(probe_model)
print("Total parameters:", n_params)
print("Trainable parameters:", n_trainable)

assert n_params > 0
assert n_params == n_trainable

print("Locked RQ1 GRU instantiated successfully.")

GRUPredictor(
  (gru): GRU(1, 256, num_layers=3, batch_first=True, dropout=0.2)
  (head): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=5, bias=True)
  )
)
Total parameters: 1021957
Trainable parameters: 1021957
Locked RQ1 GRU instantiated successfully.


## 6. Verify that one local training pass changes parameters


In [19]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, target = batch
    return model(cgm.to(device)), target.to(device)

first_pid = next(iter(client_loaders))
first_batch = next(iter(client_loaders[first_pid]))

probe_model = model_fn().to(DEVICE)
probe_model.eval()

with torch.no_grad():
    probe_pred, probe_target = forward_fn(probe_model, first_batch, DEVICE)

print("First client:", first_pid)
print("Prediction shape:", tuple(probe_pred.shape))
print("Target shape:", tuple(probe_target.shape))

assert probe_pred.shape == probe_target.shape
assert probe_pred.ndim == 2
assert probe_pred.shape[1] == 5
assert torch.isfinite(probe_pred).all()
assert torch.isfinite(probe_target).all()

print("Real client batch passes through the GRU correctly.")
print("Five forecast horizons detected.")


First client: CGMacros-001
Prediction shape: (256, 5)
Target shape: (256, 5)
Real client batch passes through the GRU correctly.
Five forecast horizons detected.


## 7. Run five FedAvg rounds

In [20]:

federated.set_seed(SEED)
initial_model = model_fn()
initial_state = {
    k: v.detach().cpu().clone()
    for k, v in initial_model.state_dict().items()
}

federated.set_seed(SEED)

model, history = fed.federated_train(
    model_fn,
    client_loaders,
    val_loader,
    stats['cgm_mean'],
    stats['cgm_std'],
    device=DEVICE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    local_epochs=LOCAL_EPOCHS,
    max_rounds=MAX_ROUNDS,

    clients_per_round=None,

    early_stop_patience=MAX_ROUNDS + 1,
    lr_patience=MAX_ROUNDS + 1,

    forward_fn=forward_fn,
    checkpoint_dir=None,
    verbose=True,
    progress_bar=False,
)

print("fed.federated_train returned successfully.")


device: cuda  model: GRUPredictor  params: 1,021,957  trainable: 1,021,957  clients: 30 (full per round)
round   1/5  val_loss=0.8923  val_rmse(mg/dL)=[39.7, 39.6, 39.8, 39.8, 40.0]  lr=1.00e-04
round   2/5  val_loss=0.8657  val_rmse(mg/dL)=[38.7, 38.8, 39.3, 39.5, 39.7]  lr=1.00e-04
round   3/5  val_loss=0.8266  val_rmse(mg/dL)=[37.2, 37.6, 38.4, 39.1, 39.1]  lr=1.00e-04
round   4/5  val_loss=0.7698  val_rmse(mg/dL)=[35.1, 35.8, 37.1, 38.3, 38.4]  lr=1.00e-04
round   5/5  val_loss=0.7025  val_rmse(mg/dL)=[32.2, 33.5, 35.4, 37.4, 37.7]  lr=1.00e-04
fed.federated_train returned successfully.


## 8. Automatic post-run checks


In [21]:
print("History keys:", list(history.keys()))

assert 'val_loss' in history, "history is missing val_loss"
assert 'val_rmse' in history, "history is missing val_rmse"

val_loss = np.asarray(history['val_loss'], dtype=float)
val_rmse = np.asarray(history['val_rmse'], dtype=float)

assert len(val_loss) == MAX_ROUNDS, (
    f"Expected {MAX_ROUNDS} rounds, got {len(val_loss)}"
)

assert np.all(np.isfinite(val_loss)), "NaN/Inf detected in validation loss."
assert np.all(np.isfinite(val_rmse)), "NaN/Inf detected in validation RMSE."

assert val_rmse.ndim == 2, f"Expected 2-D val_rmse, got {val_rmse.shape}"
assert val_rmse.shape == (MAX_ROUNDS, 5), (
    f"Expected RMSE shape ({MAX_ROUNDS}, 5), got {val_rmse.shape}"
)

final_state = {
    k: v.detach().cpu().clone()
    for k, v in model.state_dict().items()
}

changed_keys = [
    k for k in initial_state
    if torch.is_floating_point(initial_state[k])
    and not torch.equal(initial_state[k], final_state[k])
]

assert len(changed_keys) > 0, (
    "No floating-point model parameter changed. "
    "Local training or aggregation may not be updating the global model."
)

print("✓ All validation values are finite.")
print("✓ Five forecast horizons are present.")
print(f"✓ Global model changed ({len(changed_keys)} floating-point state entries differ from initialization).")


History keys: ['val_loss', 'val_rmse', 'val_mae', 'lr']
✓ All validation values are finite.
✓ Five forecast horizons are present.
✓ Global model changed (16 floating-point state entries differ from initialization).


In [23]:
print("\nValidation loss by round:")
for r, loss in enumerate(val_loss, start=1):
    print(f"  Round {r:02d}: {loss:.6f}")

print("\nValidation RMSE by round:")
for r, rmses in enumerate(val_rmse, start=1):
    formatted = ", ".join(f"{x:.3f}" for x in rmses)
    print(f"  Round {r:02d}: [{formatted}]")

print("\nInitial validation loss:", float(val_loss[0]))
print("Best validation loss:", float(val_loss.min()))

if val_loss.min() < val_loss[0]:
    print("Validation loss improved at least once during the five-round sanity run.")
else:
    print(
        "Validation loss did not improve within five rounds. "
        "This alone does not prove a bug, but inspect the trajectory before FL tuning."
    )


Validation loss by round:
  Round 01: 0.892269
  Round 02: 0.865732
  Round 03: 0.826565
  Round 04: 0.769822
  Round 05: 0.702522

Validation RMSE by round:
  Round 01: [39.701, 39.570, 39.820, 39.837, 40.029]
  Round 02: [38.735, 38.781, 39.275, 39.521, 39.658]
  Round 03: [37.249, 37.588, 38.415, 39.057, 39.147]
  Round 04: [35.054, 35.787, 37.058, 38.348, 38.434]
  Round 05: [32.236, 33.475, 35.412, 37.367, 37.727]

Initial validation loss: 0.8922685384750366
Best validation loss: 0.702521800994873
Validation loss improved at least once during the five-round sanity run.
